# 13 · Stable Diffusion Inpainting (검은 마스크 → 진짜 변형된 얼굴)

**목적**: SC-FEGAN의 TF 1.x 환경 제약을 우회. 동일 task(mask-based facial inpainting)를 **Stable Diffusion Inpaint (Rombach et al. 2022)**로 수행.

**파이프라인**:
```
사진 → 랜드마크 → 시술별 mask → 시술별 prompt → SD Inpaint → Refinement → 최종
```

**환경**: Colab T4 GPU + PyTorch + diffusers (Python 3.12 안정 호환)

**산출물**: `/MyDrive/cv-final-ckpts/samples/sd_inpaint_{procedure}.png`

**학술 framing**: SC-FEGAN(2019)은 TF 1.15 환경 의존성으로 Python 3.12에서 inference 불가. 동일 task를 더 최신 모델(SD Inpaint)로 대체하여 본 프로젝트의 **모듈식 설계**(시술 추천, 자동 입력 생성, refinement)의 강점을 검증.

## 1. 환경 셋업

In [ ]:
import os, sys, subprocess
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    REPO = '/content/cv-final'
    if not os.path.exists(REPO):
        from google.colab import userdata
        token = userdata.get('GH_TOKEN').strip()
        result = subprocess.run(
            ['git', 'clone', f'https://{token}@github.com/keonU206/cv-final.git', REPO],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f'clone 실패: {result.stderr}')
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    
    print('=== 의존성 설치 ===')
    !pip install -q diffusers transformers accelerate scipy mediapipe opencv-python pyyaml segmentation_models_pytorch
    
    import torch
    print(f'\n✅ PyTorch {torch.__version__}')
    print(f'✅ CUDA: {torch.cuda.is_available()}')
    print(f'✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 2. Stable Diffusion Inpainting 모델 로드

**모델**: `stabilityai/stable-diffusion-2-inpainting` (5GB, 첫 다운로드 2-5분)

T4 GPU에서 float16으로 메모리 최적화.

In [ ]:
import torch
from diffusers import StableDiffusionInpaintPipeline, DPMSolverMultistepScheduler

# HF 인증 (선택) — Colab Secrets에 HF_TOKEN 있으면 자동 로그인
try:
    from huggingface_hub import login
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token.strip())
        print('✅ HF 로그인 (gated 모델 접근 가능)')
except Exception:
    print('ℹ HF_TOKEN secret 없음 → 무인증 모델만 시도')

# 여러 모델 자동 시도 (안정성 순)
MODELS = [
    'runwayml/stable-diffusion-inpainting',
    'stabilityai/stable-diffusion-2-inpainting',
    'botp/stable-diffusion-v1-5-inpainting',
    'Lykon/dreamshaper-8-inpainting',
]

pipe = None
for model_id in MODELS:
    print(f'\n시도: {model_id}')
    try:
        pipe = StableDiffusionInpaintPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            safety_checker=None,
            requires_safety_checker=False,
        ).to('cuda')
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
        pipe.enable_attention_slicing()
        print(f'  ✅ 로드 성공: {model_id}')
        MODEL_ID = model_id
        break
    except Exception as e:
        msg = str(e)[:120]
        print(f'  ❌ {type(e).__name__}: {msg}')

if pipe is None:
    raise RuntimeError(
        'SD inpainting 모델 로드 실패. HF_TOKEN 등록 후 재시도:\n'
        '  1. https://huggingface.co/settings/tokens 에서 Read 권한 토큰 발급\n'
        '  2. Colab Secrets에 HF_TOKEN 추가 (Notebook access ON)\n'
        '  3. 모델 페이지에서 license agree (해당 모델만)\n'
        '  4. 이 셀 다시 실행'
    )

## 3. 사진 + 랜드마크 추출

⚠ **MediaPipe 0.10.35+ 호환** — `mp.solutions`는 deprecated, **`mp.tasks.vision.FaceLandmarker`** 사용.

처음 실행 시 `face_landmarker.task` 모델 (8MB) 자동 다운로드.

업로드 사진은 **정면, 영문 파일명** 권장 (`face.jpg` 등).

In [ ]:
import numpy as np, cv2, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

SIZE = 512
OUTPUT_DIR = Path('outputs/sd_inpaint')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from google.colab import files
    print('얼굴 사진 1장 업로드 (정면, 영문 파일명 권장):')
    uploaded = files.upload()
    IMG_PATH = list(uploaded.keys())[0]
else:
    IMG_PATH = 'demo_face.jpg'

image_rgb = cv2.cvtColor(cv2.imread(IMG_PATH), cv2.COLOR_BGR2RGB)
image_rgb = cv2.resize(image_rgb, (SIZE, SIZE))

# ─── MediaPipe Tasks API (0.10.35+) ───
# mp.solutions는 0.10.35에서 제거됨 → mp.tasks 사용
LANDMARKER_MODEL = '/content/face_landmarker.task'
if not os.path.exists(LANDMARKER_MODEL):
    print('Face Landmarker 모델 다운로드 (8MB)...')
    !wget -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task -O {LANDMARKER_MODEL}

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

base_options = mp_python.BaseOptions(model_asset_path=LANDMARKER_MODEL)
options = mp_vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1,
)
detector = mp_vision.FaceLandmarker.create_from_options(options)

mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
result = detector.detect(mp_image)

if not result.face_landmarks:
    raise RuntimeError('얼굴 감지 실패')

lms = result.face_landmarks[0]
landmarks = np.array([[lm.x * SIZE, lm.y * SIZE] for lm in lms], dtype=np.int32)
print(f'✅ Landmarks: {landmarks.shape} (478점 with refine)')

plt.figure(figsize=(5, 5))
plt.imshow(image_rgb); plt.axis('off'); plt.title('Input')
plt.show()

## 4. 시술별 mask + prompt 설정

In [ ]:
from project.input_generator import make_mask, load_procedures

procedures_db = load_procedures()

# ─── 개선된 SD 프롬프트 (identity 보존 + 자연스러움 강조) ───
# 핵심: "same person, same face" 키워드로 SD가 다른 사람으로 바꾸지 않게 강제
COMMON_NEG = 'different person, different face, race change, ethnic change, deformed, ugly, blurry, low quality, distorted, cartoon, anime, painting, sketch, oversaturated, makeup heavy, asymmetric, bad anatomy, watermark, text'

PROMPTS = {
    'nose_tip': {
        'positive': 'same person, same face, only refined nose tip, slightly slimmer nose, subtle natural change, Korean beauty, photorealistic portrait, high quality skin texture, soft natural lighting, professional photography, RAW photo',
        'negative': COMMON_NEG + ', large nose change, fake nose, plastic surgery look',
    },
    'double_eyelid': {
        'positive': 'same person, same face, only subtle double eyelid added, natural eye crease, bright open eyes, Korean beauty, photorealistic portrait, high quality, RAW photo',
        'negative': COMMON_NEG + ', closed eyes, heavy eyeshadow, eyeliner, fake lashes',
    },
    'jaw_v_line': {
        'positive': 'same person, same face, only subtle slimmer jawline, refined chin, natural V-line, Korean beauty, photorealistic portrait, high quality skin, RAW photo',
        'negative': COMMON_NEG + ', wide jaw, square face, masculine jaw, fake chin',
    },
}

# ─── Mask 영역 좁히기 (dilate 감소로 변형 영역 보수적) ───
# procedures.yaml의 dilate_px를 동적으로 줄이기 (자연스러운 경계)
MASK_DILATE_OVERRIDE = {
    'nose_tip': 8,        # 원래 18 → 8 (좁은 변형)
    'double_eyelid': 6,   # 원래 10 → 6
    'jaw_v_line': 15,     # 원래 25 → 15
}

masks = {}
for proc_id in PROMPTS.keys():
    proc = procedures_db[proc_id].copy()
    proc['dilate_px'] = MASK_DILATE_OVERRIDE[proc_id]  # override
    mask_arr = make_mask(landmarks, proc, size=SIZE)
    masks[proc_id] = mask_arr[..., 0]
    print(f'{proc_id}: mask 픽셀 {(mask_arr > 0).sum():,} (dilate={proc["dilate_px"]})')

## 5. Stable Diffusion Inpainting 실행

In [ ]:
import torch, time

def np_to_pil(arr):
    return Image.fromarray(arr)

image_pil = np_to_pil(image_rgb)

# ─── 개선된 inference 파라미터 ───
# steps 30 → 50: 더 정교한 결과
# guidance 7.5 → 7.0: 약간 더 자연스럽게
# strength: 변형 강도 조절 (낮을수록 원본 보존)
INFERENCE_CONFIG = {
    'num_inference_steps': 50,
    'guidance_scale': 7.0,
    # strength는 일부 모델만 지원 — try/except로 안전 처리
}

results_sd = {}
total_start = time.time()

for proc_id, prompt_info in PROMPTS.items():
    print(f'\n━━━ {proc_id} ━━━')
    print(f'  prompt: "{prompt_info["positive"][:70]}..."')
    print(f'  inpainting 중 (50 steps, T4 GPU)...')
    
    mask_pil = np_to_pil(masks[proc_id])
    t0 = time.time()
    
    result = pipe(
        prompt=prompt_info['positive'],
        negative_prompt=prompt_info['negative'],
        image=image_pil,
        mask_image=mask_pil,
        num_inference_steps=INFERENCE_CONFIG['num_inference_steps'],
        guidance_scale=INFERENCE_CONFIG['guidance_scale'],
        generator=torch.manual_seed(42),
    ).images[0]
    
    elapsed = time.time() - t0
    results_sd[proc_id] = np.array(result)
    print(f'  ✅ 완료 ({elapsed:.1f}초) · shape: {results_sd[proc_id].shape}')

total_elapsed = time.time() - total_start
print(f'\n=== 전체 inference 완료 ({total_elapsed:.1f}초) ===')
print(f'\n💡 결과가 여전히 어색하면:')
print(f'  - seed 변경 시도 (현재 42)')
print(f'  - prompt 더 강하게 (예: "VERY subtle, minimal change")')
print(f'  - MASK_DILATE_OVERRIDE 더 줄이기 (셀 4)')

## 6. Refinement 적용

In [ ]:
import torch
from project.refinement.model import build_refinement_wrapper

device = 'cuda' if torch.cuda.is_available() else 'cpu'
REF_CKPT = '/content/drive/MyDrive/cv-final-ckpts/refinement_best.pt'

if os.path.exists(REF_CKPT):
    ref_model = build_refinement_wrapper(encoder_weights=None).to(device)
    ref_model.load_state_dict(torch.load(REF_CKPT, map_location=device))
    ref_model.eval()
    print(f'✅ Refinement 로드: {REF_CKPT}')
else:
    ref_model = None
    print('⚠ Refinement ckpt 없음 → SD 결과 그대로 사용')

def refine(img_rgb):
    if ref_model is None:
        return img_rgb
    tensor = torch.from_numpy(img_rgb).float() / 127.5 - 1.0
    tensor = tensor.permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        out = ref_model(tensor)[0].cpu()
    return ((out.clamp(-1, 1) + 1) * 127.5).byte().permute(1, 2, 0).numpy()

results_final = {}
for proc_id, sd_out in results_sd.items():
    results_final[proc_id] = refine(sd_out)
    print(f'  {proc_id}: refined')

## 7. 결과 시각화 (4컬럼 비교)

In [ ]:
# 시술 한글명 매핑 (발표 슬라이드용)
PROC_KO = {
    'nose_tip': '코끝 성형',
    'double_eyelid': '쌍커풀',
    'jaw_v_line': 'V라인 (사각턱)',
}

# === 1) 종합 4컬럼 비교 (3 시술 × 4 컬럼) ===
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
for i, proc_id in enumerate(PROMPTS.keys()):
    name = PROC_KO[proc_id]
    
    axes[i, 0].imshow(image_rgb)
    axes[i, 0].set_title(f'{name}\nBefore (원본)', fontsize=13)
    axes[i, 1].imshow(masks[proc_id], cmap='gray')
    axes[i, 1].set_title('Mask\n(시술 영역)', fontsize=13)
    axes[i, 2].imshow(results_sd[proc_id])
    axes[i, 2].set_title('SD Inpaint ⭐\n(GAN 결과)', fontsize=13)
    axes[i, 3].imshow(results_final[proc_id])
    axes[i, 3].set_title('+ Refinement\n(최종)', fontsize=13)
    for ax in axes[i]:
        ax.axis('off')

plt.tight_layout()
save_all = OUTPUT_DIR / 'all_sd_results.png'
plt.savefig(save_all, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ 종합 비교: {save_all}')

# === 2) 시술별 개별 PNG (발표 슬라이드 1장당 1개) ===
for proc_id in PROMPTS.keys():
    name = PROC_KO[proc_id]
    
    # Before / After 단순 비교 (발표용)
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(image_rgb)
    axes[0].set_title(f'{name} · Before', fontsize=16)
    axes[1].imshow(results_final[proc_id])
    axes[1].set_title(f'{name} · After', fontsize=16)
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    save_proc = OUTPUT_DIR / f'beforeafter_{proc_id}.png'
    plt.savefig(save_proc, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'  ✅ 발표용 Before/After: {save_proc}')
    
    # 결과만 (Streamlit/PDF 활용)
    cv2.imwrite(
        str(OUTPUT_DIR / f'sd_final_{proc_id}.png'),
        cv2.cvtColor(results_final[proc_id], cv2.COLOR_RGB2BGR),
    )
    cv2.imwrite(
        str(OUTPUT_DIR / f'sd_only_{proc_id}.png'),
        cv2.cvtColor(results_sd[proc_id], cv2.COLOR_RGB2BGR),
    )

# 원본도 저장 (PDF 견적서용)
cv2.imwrite(
    str(OUTPUT_DIR / 'original_before.png'),
    cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR),
)
print(f'\n총 저장 파일: {len(list(OUTPUT_DIR.glob("*.png")))} 개')
print(f'위치: {OUTPUT_DIR.resolve()}')

## 7-B. 🌟 통합 변형 (3개 시술 한 번에 적용) ⭐ 발표 핵심

3가지 시술 (코끝 + 쌍커풀 + V라인)을 **순차 누적 적용**하여 최종 통합 결과 생성.

**두 가지 방식**:
1. **Union Mask** — 3개 mask 합쳐서 1회 inference (빠름, 30초)
2. **Sequential** — 시술별 순차 적용 (단계 변화 시각화, 90초)

→ 발표 슬라이드 핵심: "**한 사진 → 종합 시뮬레이션 결과**"

In [ ]:
# ═══════════════════════════════════════════════════
# 방식 1: UNION MASK — 3개 mask 합쳐서 1회 inference
# ═══════════════════════════════════════════════════

# 모든 mask 합치기 (bitwise OR)
union_mask = np.zeros((SIZE, SIZE), dtype=np.uint8)
for proc_id in PROMPTS.keys():
    union_mask = cv2.bitwise_or(union_mask, masks[proc_id])

print(f'Union mask 영역: {(union_mask > 0).sum():,} 픽셀')

# 통합 prompt (3가지 변형 동시 명시)
COMBINED_PROMPT = (
    "same person, same face, "
    "refined nose tip with slim natural bridge, "
    "natural double eyelid with subtle crease, "
    "slim v-line jaw with refined chin, "
    "Korean beauty, photorealistic portrait, "
    "high quality skin texture, soft natural lighting, "
    "RAW photo, detailed facial features"
)
COMBINED_NEGATIVE = (
    COMMON_NEG + ", "
    "wide jaw, square face, large nose, closed eyes, "
    "asymmetric eyes, plastic surgery look"
)

print('\n━━━ Union 방식 inpainting ━━━')
t0 = time.time()
union_mask_pil = np_to_pil(union_mask)
union_result = pipe(
    prompt=COMBINED_PROMPT,
    negative_prompt=COMBINED_NEGATIVE,
    image=image_pil,
    mask_image=union_mask_pil,
    num_inference_steps=50,
    guidance_scale=7.0,
    generator=torch.manual_seed(42),
).images[0]
union_arr = np.array(union_result)
print(f'  ✅ 완료 ({time.time() - t0:.1f}초)')

# Refinement 적용
union_refined = refine(union_arr) if ref_model else union_arr

In [ ]:
# ═══════════════════════════════════════════════════
# 방식 2: SEQUENTIAL — 시술 순차 누적 적용 (단계별 변화)
# ═══════════════════════════════════════════════════

print('━━━ Sequential 방식 inpainting (3개 시술 누적) ━━━')
print('  시술 적용 순서: 코끝 → 쌍커풀 → V라인')

sequential_stages = {}  # 각 단계별 결과 저장
sequential_stages['original'] = image_rgb.copy()

current_image = image_rgb.copy()
current_pil = np_to_pil(current_image)

stage_order = ['nose_tip', 'double_eyelid', 'jaw_v_line']
for proc_id in stage_order:
    prompt_info = PROMPTS[proc_id]
    mask_pil = np_to_pil(masks[proc_id])
    name = PROC_KO[proc_id]
    
    print(f'\n  Step: + {name}')
    t0 = time.time()
    result = pipe(
        prompt=prompt_info['positive'],
        negative_prompt=prompt_info['negative'],
        image=current_pil,
        mask_image=mask_pil,
        num_inference_steps=50,
        guidance_scale=7.0,
        generator=torch.manual_seed(42),
    ).images[0]
    
    current_image = np.array(result)
    current_pil = result
    sequential_stages[proc_id] = current_image.copy()
    print(f'    ✅ {time.time() - t0:.1f}초')

# 최종 누적 결과
sequential_final = sequential_stages[stage_order[-1]]
sequential_refined = refine(sequential_final) if ref_model else sequential_final

print(f'\n=== Sequential 완료: 총 {len(stage_order)}단계 ===')

In [ ]:
# ═══════════════════════════════════════════════════
# 시각화 ⭐ 발표 핵심 슬라이드
# ═══════════════════════════════════════════════════

# === Sequential 단계별 변화 (1 row × 5 cols) ===
fig, axes = plt.subplots(1, 5, figsize=(25, 5.5))
stage_titles = [
    ('original', 'Before (원본)'),
    ('nose_tip', '+ 코끝 성형'),
    ('double_eyelid', '+ 쌍커풀'),
    ('jaw_v_line', '+ V라인'),
]

for i, (key, title) in enumerate(stage_titles):
    axes[i].imshow(sequential_stages[key])
    axes[i].set_title(title, fontsize=15, fontweight='bold')
    axes[i].axis('off')

# 마지막: Refinement 적용 최종
axes[4].imshow(sequential_refined)
axes[4].set_title('+ Refinement\n(최종)', fontsize=15, fontweight='bold')
axes[4].axis('off')

plt.suptitle('🌟 통합 시술 시뮬레이션 — 코끝 → 쌍커풀 → V라인 → Refinement',
             fontsize=17, y=1.02, fontweight='bold')
plt.tight_layout()
save_seq = OUTPUT_DIR / 'sequential_stages.png'
plt.savefig(save_seq, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ Sequential 단계별: {save_seq}')

# === Before vs After (Union vs Sequential 비교) ===
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
axes[0].imshow(image_rgb)
axes[0].set_title('Before (원본)', fontsize=15, fontweight='bold')
axes[1].imshow(union_mask, cmap='gray')
axes[1].set_title('통합 Mask (3개)', fontsize=15, fontweight='bold')
axes[2].imshow(union_refined)
axes[2].set_title('After · Union 방식\n(1회 inference)', fontsize=15, fontweight='bold')
axes[3].imshow(sequential_refined)
axes[3].set_title('After · Sequential 방식 ⭐\n(누적 3회)', fontsize=15, fontweight='bold')
for ax in axes:
    ax.axis('off')

plt.suptitle('통합 변형 결과 — Union vs Sequential', fontsize=17, y=1.02, fontweight='bold')
plt.tight_layout()
save_combined = OUTPUT_DIR / 'combined_all_procedures.png'
plt.savefig(save_combined, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ Union vs Sequential: {save_combined}')

# === Before/After 단순 (발표용) ===
fig, axes = plt.subplots(1, 2, figsize=(12, 6.5))
axes[0].imshow(image_rgb)
axes[0].set_title('Before\n(원본)', fontsize=18, fontweight='bold')
axes[1].imshow(sequential_refined)
axes[1].set_title('After\n(코끝 + 쌍커풀 + V라인)', fontsize=18, fontweight='bold')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
save_ba_combined = OUTPUT_DIR / 'beforeafter_combined.png'
plt.savefig(save_ba_combined, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ 발표용 Before/After 통합: {save_ba_combined}')

# 결과만 저장 (PDF/슬라이드용)
cv2.imwrite(
    str(OUTPUT_DIR / 'sd_final_combined.png'),
    cv2.cvtColor(sequential_refined, cv2.COLOR_RGB2BGR),
)
cv2.imwrite(
    str(OUTPUT_DIR / 'sd_union_combined.png'),
    cv2.cvtColor(union_refined, cv2.COLOR_RGB2BGR),
)
print('\n💡 발표 시 사용 권장:')
print('  - beforeafter_combined.png → 메인 슬라이드 (Before/After 단순)')
print('  - sequential_stages.png → 단계별 변화 (학술 깊이)')
print('  - combined_all_procedures.png → Union/Sequential 비교 (방법론)')

## 8. Drive 백업

In [ ]:
if IS_COLAB:
    DRIVE_OUT = Path('/content/drive/MyDrive/cv-final-ckpts/samples')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    
    # 복사 + 결과 확인
    !cp -v {OUTPUT_DIR}/*.png {DRIVE_OUT}/ 2>&1 | tail -15
    
    print(f'\n✅ Drive 백업 완료: {DRIVE_OUT}')
    print(f'\n=== 백업된 파일 목록 ===')
    !ls -lh {DRIVE_OUT}/ | grep -E "sd_|beforeafter|original" | tail -15

print('\n' + '=' * 60)
print('=== Phase 7-G 완료 ===')
print('=' * 60)
print('\n📦 산출물 활용 가이드:')
print('  1. 발표 슬라이드:')
print('     - all_sd_results.png  → 메인 결과 슬라이드 (4컬럼 비교)')
print('     - beforeafter_*.png   → 시술별 1슬라이드씩 (Before/After)')
print('  2. Streamlit 데모:')
print('     - sd_final_*.png      → 사이드바 결과 표시')
print('     - original_before.png → 입력 자리')
print('  3. PDF 견적서:')
print('     - sd_final_*.png + original_before.png 로 build_estimate() 호출')
print('\n💡 다음 단계:')
print('  - 로컬로 다운로드: 위 PNG들을 /MyDrive/cv-final-ckpts/samples 에서 받기')
print('  - Streamlit 통합: app/streamlit_demo.py 에서 정적 이미지 표시 옵션 추가')

---

## 발표 framing (학술 정직성)

**기술 선택 정당화**:
1. 원 계획: SC-FEGAN (Jo & Park 2019, ICCV)
2. 문제: TF 1.15 의존, Python 3.12 환경에서 inference 불가
3. 우회: **동일 task(mask-based facial inpainting)를 수행하는 최신 모델 SD Inpaint (Rombach et al. 2022, CVPR)** 채택
4. 장점:
   - 더 최신 기술 (2022 vs 2019)
   - 환경 안정성 (PyTorch 기반)
   - 우리 모듈식 설계의 강점 검증 (시술 추천, 자동 입력 생성, Refinement는 model-agnostic)

**참고문헌**:
- Rombach, R. et al. (2022). High-Resolution Image Synthesis with Latent Diffusion Models. CVPR.
- Jo, Y., & Park, J. (2019). SC-FEGAN: Face Editing GAN with User's Sketch and Color. ICCV.